# Vision Transformer for CIFAR-10 Image Classification

> **An Image is Worth 16×16 Words** · Dosovitskiy et al. (2020)
>
> This notebook walks through the entire pipeline:
> 1. Dataset loading & visualisation
> 2. ViT model architecture
> 3. Training
> 4. Evaluation & result plots
> 5. Single-image prediction demo

## 1. Setup & Imports

In [ ]:
import os, sys
# Add project root to path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from utils.config import DEVICE, CLASS_NAMES, IMAGE_SIZE

print(f'PyTorch version : {torch.__version__}')
print(f'Device          : {DEVICE}')
print(f'CIFAR-10 classes: {CLASS_NAMES}')

## 2. Dataset Loading & Visualisation

In [ ]:
from dataset.cifar10_loader import get_dataloaders

train_loader, val_loader, test_loader = get_dataloaders(batch_size=32)
images, labels = next(iter(train_loader))
print(f'Image batch : {images.shape}')  # (32, 3, 224, 224)
print(f'Label batch : {labels.shape}')  # (32,)

In [ ]:
# Visualise a sample batch
from utils.config import NORMALIZE_MEAN, NORMALIZE_STD
import torch

mean = torch.tensor(NORMALIZE_MEAN).view(1, 3, 1, 1)
std  = torch.tensor(NORMALIZE_STD).view(1, 3, 1, 1)
imgs_show = (images[:16] * std + mean).clamp(0, 1).permute(0, 2, 3, 1).numpy()

fig, axes = plt.subplots(4, 4, figsize=(10, 10))
for i, ax in enumerate(axes.flatten()):
    ax.imshow(imgs_show[i])
    ax.set_title(CLASS_NAMES[labels[i].item()], fontsize=10)
    ax.axis('off')
plt.suptitle('CIFAR-10 Sample Batch (rescaled to 224×224)', fontsize=13)
plt.tight_layout()
plt.show()

## 3. Model Architecture

In [ ]:
from models.vision_transformer import vit_tiny, vit_small, VisionTransformer

model = vit_tiny()
print(f'Total trainable parameters: {model.get_num_params():,}')
print()
print(model)

In [ ]:
# Forward pass sanity check
dummy = torch.randn(2, 3, IMAGE_SIZE, IMAGE_SIZE)
with torch.no_grad():
    out = model(dummy)
print(f'Input  shape  : {dummy.shape}')   # (2, 3, 224, 224)
print(f'Output shape  : {out.shape}')     # (2, 10)
print(f'Output logits : {out[0].tolist()}')

## 4. Training

In [ ]:
# Run a quick training demo (2 epochs)
from training.train import train

history = train(
    model_variant='tiny',
    num_epochs=2,       # Increase to 20+ for real training
    batch_size=64,
)

In [ ]:
# Plot training history inline
epochs = range(1, len(history['train_losses']) + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(epochs, history['train_losses'], label='Train Loss', color='#3B82F6')
ax1.plot(epochs, history['val_losses'],   label='Val Loss',   color='#EF4444', linestyle='--')
ax1.set_title('Loss Curve')
ax1.set_xlabel('Epoch')
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(epochs, history['train_accs'], label='Train Acc', color='#10B981')
ax2.plot(epochs, history['val_accs'],   label='Val Acc',   color='#F59E0B', linestyle='--')
ax2.set_title('Accuracy Curve')
ax2.set_xlabel('Epoch')
ax2.legend()
ax2.grid(alpha=0.3)

plt.suptitle('Training History', fontsize=13)
plt.tight_layout()
plt.show()

## 5. Evaluation

In [ ]:
from evaluation.test_model import evaluate

test_acc = evaluate(save_plots=True)
print(f'\nFinal Test Accuracy: {test_acc:.2f}%')

## 6. Single Image Prediction Demo

In [ ]:
from app.predict_image import predict
import requests
from io import BytesIO

# Download a sample dog image from the web
url = 'https://upload.wikimedia.org/wikipedia/commons/thumb/2/26/YellowLabradorLooking_new.jpg/320px-YellowLabradorLooking_new.jpg'
resp = requests.get(url, timeout=10)
img  = Image.open(BytesIO(resp.content)).convert('RGB')

# Save to a temp file
tmp = '/tmp/demo_dog.jpg'
img.save(tmp)

results = predict(tmp, topk=5)

fig, (ax_img, ax_bar) = plt.subplots(1, 2, figsize=(12, 5))

# Show image
ax_img.imshow(img.resize((224, 224)))
ax_img.set_title(f'Predicted: {results[0]["class"].upper()}\nConfidence: {results[0]["confidence"]:.1f}%',
                 fontsize=12, fontweight='bold')
ax_img.axis('off')

# Confidence bars
labels_r = [r['class'].capitalize() for r in results]
confs_r  = [r['confidence'] for r in results]
colors_r = ['#6366f1' if i == 0 else '#94a3b8' for i in range(len(results))]
ax_bar.barh(labels_r[::-1], confs_r[::-1], color=colors_r[::-1])
ax_bar.set_xlabel('Confidence (%)')
ax_bar.set_title('Top-5 Predictions')
ax_bar.set_xlim(0, 105)
ax_bar.grid(axis='x', alpha=0.3)
for i, (lbl, conf) in enumerate(zip(labels_r[::-1], confs_r[::-1])):
    ax_bar.text(conf + 1, i, f'{conf:.1f}%', va='center', fontsize=9)

plt.suptitle('Vision Transformer – Image Prediction Demo', fontsize=13)
plt.tight_layout()
plt.show()